# Course Difficulty Prediction Task

**Research:** An End-to-End Study on Prompt Engineering Techniques in Generative AI  
for Automated Course Understanding, Skill Extraction, and Personalized Learning Insights

**Setup and Execution Guide**

Step 1: Environment Setup

Install all required libraries and dependencies.

Step 2: Pipeline Configuration and Execution

This step covers dataset loading, prompt setup, model configuration, evaluation and exporting results.

CONFIG section at the top to be updated as per the need of experiments.

In [ ]:
# =============================================================================
# INSTALL DEPENDENCIES
# =============================================================================

import subprocess
import sys

SEPARATOR_WIDTH: int = 55

LIBRARY_INSTALLS = [
    ("openai>=1.30.0",),
    ("pandas>=2.0.0", "numpy>=1.24.0", "scipy>=1.10.0"),
    ("matplotlib>=3.7.0",),
    ("scikit-learn>=1.3.0",),
]

VERSION_CHECK_LIBS: list = [
    ("openai",       "openai"),
    ("pandas",       "pandas"),
    ("numpy",        "numpy"),
    ("sklearn",      "scikit-learn"),
    ("matplotlib",   "matplotlib"),
    ("scipy",        "scipy"),
]


def pip_install(*packages: str) -> None:
    """Install one or more pip packages quietly via subprocess."""
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *packages]
    )


def print_library_versions(lib_map: list) -> None:
    """Print installed version for each library in lib_map.

    Args:
        lib_map: List of (import_name, display_name) tuples.
    """
    print("=" * SEPARATOR_WIDTH)
    print("INSTALLED LIBRARY VERSIONS")
    print("=" * SEPARATOR_WIDTH)
    for import_name, display_name in lib_map:
        try:
            mod = __import__(import_name)
            version = getattr(mod, "__version__", "unknown")
            print(f"  {display_name:<18} {version}")
        except ImportError:
            print(f"  {display_name:<18} NOT INSTALLED")
    print("=" * SEPARATOR_WIDTH)


print("[1/4] Installing core packages...")
for pkg_group in LIBRARY_INSTALLS:
    pip_install(*pkg_group)

print()
print_library_versions(VERSION_CHECK_LIBS)
print()
print("All packages installed successfully.")
print("If Colab prompts a runtime restart, click Restart.")
print("After restarting, run Cell 2 directly — do NOT re-run Cell 1.")

[1/4] Installing core packages...

INSTALLED LIBRARY VERSIONS
  openai             2.32.0
  pandas             2.2.2
  numpy              2.0.2
  scikit-learn       1.6.1
  matplotlib         3.10.0
  scipy              1.16.3

All packages installed successfully.
If Colab prompts a runtime restart, click Restart.
After restarting, run Cell 2 directly — do NOT re-run Cell 1.


In [ ]:
# =============================================================================
# CELL 2 — END-TO-END DIFFICULTY PREDICTION EXPERIMENT
#
# Sections
# --------
#   A.  Imports & library version display
#   B.  *** TEMPLATE VARIABLES — edit here to create a new experiment ***
#   C.  Configuration
#   D.  Dataset load
#   E.  API key setup
#   F.  _build_prompt() — difficulty prediction prompt ***
#   G.  LLM caller
#   H.  Run experiment
#   I.  Evaluation metrics
#   J.  Full metrics table
#   K.  Final summary
#   L.  Save & download CSVs
# =============================================================================


# =============================================================================
# A.  IMPORTS & LIBRARY VERSION DISPLAY
# =============================================================================

import getpass
import math
import os
import random
import re
import time
import warnings
from collections import Counter
from datetime import datetime, timezone
from typing import Dict, List, Optional, Tuple

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
)
from openai import OpenAI

try:
    from google.colab import drive, files
    IN_COLAB: bool = True
except ImportError:
    IN_COLAB = False

matplotlib.rcParams["figure.dpi"] = 110

SEPARATOR_SM: int = 40
SEPARATOR_MD: int = 55
SEPARATOR_LG: int = 65
SEPARATOR_XL: int = 80
SEPARATOR_XXL: int = 90
TABLE_ROW_WIDTH: int = 115

LIBRARY_VERSION_MAP: List[Tuple[str, str]] = [
    ("openai",       "openai"),
    ("pandas",       "pandas"),
    ("numpy",        "numpy"),
    ("sklearn",      "scikit-learn"),
    ("matplotlib",   "matplotlib"),
    ("scipy",        "scipy"),
]


def print_library_versions(lib_map: List[Tuple[str, str]]) -> None:
    """Print the active runtime version for each library.

    Args:
        lib_map: List of (import_name, display_name) tuples.
    """
    print("=" * SEPARATOR_MD)
    print("LIBRARY VERSIONS (active runtime)")
    print("=" * SEPARATOR_MD)
    for import_name, display_name in lib_map:
        try:
            mod = __import__(import_name)
            version = getattr(mod, "__version__", "unknown")
            print(f"  {display_name:<18} {version}")
        except ImportError:
            print(f"  {display_name:<18} NOT INSTALLED")
    print("=" * SEPARATOR_MD)


print_library_versions(LIBRARY_VERSION_MAP)


# =============================================================================
# B.  *** TEMPLATE VARIABLES ***
#
#  --------------
#  TECHNIQUE        : "zero_shot" | "few_shot" | "instruction" | "role_based"
#                     "chain_of_thought" | "self_consistency" | "react" | "tree_of_thought"
#  GROQ_SECRET_NAME : the Colab Secret name that holds the API key for this slot
# =============================================================================

TECHNIQUE: str = "instruction"
TASK_TYPE: str = "difficulty_prediction"
GROQ_SECRET_NAME: str = "GROQ_INSTRUCTION"


# =============================================================================
# C.  CONFIGURATION
# =============================================================================

# Number of repeated runs  (1 = quick test | 3-5 = full research study)
NUM_RUNS: int = 1

# Temperature for classification (deterministic preferred)
TEMPERATURE: float = 0.0

TOP_P: float = 0.9

MODELS: List[str] = [
    "llama-3.1-8b-instant",
    "qwen/qwen3-32b",
    "openai/gpt-oss-20b",
    "openai/gpt-oss-120b",
    "llama-3.3-70b-versatile",
]

# Valid difficulty labels — used for normalisation and evaluation
VALID_LABELS: List[str] = ["Beginner", "Intermediate", "Advanced"]
LABEL_ORDER: Dict[str, int] = {"Beginner": 0, "Intermediate": 1, "Advanced": 2}

GROQ_BASE_URL: str = "https://api.groq.com/openai/v1"
MAX_TOKENS: int = 1024
DELAY_MIN: int = 7
DELAY_MAX: int = 12
N_ROWS: int = 25


def print_experiment_config() -> None:
    """Print a summary of the current experiment configuration."""
    total_calls = NUM_RUNS * len(MODELS) * N_ROWS
    avg_delay_min = total_calls * (DELAY_MIN + DELAY_MAX) / 2 / 60
    print("Experiment Configuration")
    print("-" * SEPARATOR_SM)
    print(f"  Technique          : {TECHNIQUE}")
    print(f"  Task type          : {TASK_TYPE}")
    print(f"  Temperature        : {TEMPERATURE}")
    print(f"  Runs               : {NUM_RUNS}")
    print(f"  Models             : {MODELS}")
    print(f"  Courses            : {N_ROWS}")
    print(f"  Valid labels       : {VALID_LABELS}")
    print(f"  Total API calls    : {total_calls}")
    print(f"  Est. delay time    : ~{avg_delay_min:.1f} min")
    print(f"  top_p              : {TOP_P}")


print_experiment_config()


# =============================================================================
# D.  DATASET LOAD
# =============================================================================

COLUMN_RENAME_MAP: Dict[str, str] = {
    "title":        "course_title",
    "Organization": "course_organization",
    "Level":        "course_difficulty",
    "Description":  "description",
    "enrolled":     "course_students_enrolled",
    "rating":       "course_rating",
}

TEXT_COLUMNS: List[str] = [
    "course_title",
    "course_organization",
    "course_difficulty",
    "description",
]


def load_dataset(n_rows: int = N_ROWS) -> pd.DataFrame:
    """Load, rename, and clean the Coursera CSV dataset from Google Drive.
    Merges gold difficulty labels from the gold_difficulty CSV.

    Args:
        n_rows: Maximum number of rows to load.

    Returns:
        Cleaned DataFrame with standardised columns and gold_difficulty column.

    Raises:
        FileNotFoundError: If the dataset CSV is not found at the expected path.
    """
    print("\n" + "-" * SEPARATOR_SM)
    print("Dataset Load")
    print("-" * SEPARATOR_SM)
    print("  Loading Coursera dataset from Google Drive...")
    print("  Please wait...\n")

    drive.mount("/content/drive")

    dataset_path = "/content/drive/MyDrive/research/data/coursera_clean_sample.csv"
    gold_csv_path = "/content/drive/MyDrive/research/data/gold_difficulty_coursera.csv"

    if not os.path.exists(dataset_path):
        raise FileNotFoundError(
            f"Dataset not found at '{dataset_path}'.\n"
            "Please ensure the file exists in your Google Drive."
        )

    print(f"  Found dataset at: {dataset_path}")
    print("  Cleaning and filtering rows...")

    df = pd.read_csv(dataset_path, encoding="latin-1")
    df = df.rename(columns=COLUMN_RENAME_MAP)
    df = df.dropna(subset=["course_title"]).head(n_rows).reset_index(drop=True)

    for col in TEXT_COLUMNS:
        if col in df.columns:
            df[col] = df[col].fillna("N/A").astype(str)

    print(f"  Loaded {len(df)} courses successfully.")

    # Load gold difficulty labels
    if not os.path.exists(gold_csv_path):
        raise FileNotFoundError(
            f"Gold difficulty CSV not found at '{gold_csv_path}'.\n"
            "Expected file: gold_difficulty_coursera.csv in your Drive."
        )

    print(f"  Loading gold difficulty labels from: {gold_csv_path}")
    gold_df = pd.read_csv(gold_csv_path, encoding="latin-1")

    # Identify gold label column
    gold_col_candidates = ["gold_difficulty", "difficulty", "Level", "level"]
    gold_col = next(
        (c for c in gold_col_candidates if c in gold_df.columns), None
    )
    if gold_col is None:
        text_cols = [
            c for c in gold_df.columns
            if gold_df[c].dtype == object and c not in ("course_title",)
        ]
        if not text_cols:
            raise ValueError(
                f"Cannot find a gold difficulty column in {gold_csv_path}.\n"
                f"Expected one of {gold_col_candidates} or a text column."
            )
        gold_col = text_cols[0]
        print(f"  Warning: using fallback gold column '{gold_col}'")

    print(f"  Gold difficulty column : '{gold_col}'")

    if "course_title" in gold_df.columns:
        df = df.merge(
            gold_df[["course_title", gold_col]].rename(
                columns={gold_col: "gold_difficulty"}
            ),
            on="course_title",
            how="left",
        )
    else:
        df["gold_difficulty"] = gold_df[gold_col].values[: len(df)]

    missing = df["gold_difficulty"].isna().sum()
    if missing:
        print(
            f"  Warning: {missing} course(s) have no gold difficulty label — "
            "they will be skipped in evaluation."
        )
    print(f"  Gold labels loaded : {df['gold_difficulty'].notna().sum()}")
    print(f"  Label distribution :\n{df['gold_difficulty'].value_counts().to_string()}")

    return df


def get_course_context(row: pd.Series) -> Dict[str, str]:
    """Convert a dataset row into a context dict for prompt templates.

    Args:
        row: A single row from the dataset DataFrame.

    Returns:
        Dictionary with keys: title, organization, difficulty, description,
        gold_difficulty.
    """
    return {
        "title":           str(row.get("course_title", "Unknown")),
        "organization":    str(row.get("course_organization", "N/A")),
        "description":     str(row.get("description", "No description available.")),
        "gold_difficulty": str(row.get("gold_difficulty", "")),
    }


DATASET_DF: pd.DataFrame = load_dataset(N_ROWS)
print("Loaded rows:", len(DATASET_DF))


# =============================================================================
# E.  API KEY SETUP
# =============================================================================

def load_groq_api_key(secret_name: str) -> str:
    """Load the Groq API key from Colab Secrets, falling back to getpass.

    Args:
        secret_name: The Colab Secret key name (e.g. "GROQ_ZEROSHOT").

    Returns:
        Groq API key string.

    Raises:
        ValueError: If no key is provided or found.
    """
    print("\n" + "-" * SEPARATOR_SM)
    print("API Key Setup")
    print("-" * SEPARATOR_SM)

    api_key = ""

    if IN_COLAB:
        try:
            from google.colab import userdata
            api_key = userdata.get(secret_name).strip()
            print(f"  Loaded key from Colab Secret '{secret_name}'.")
        except Exception:
            print(
                f"  Secret '{secret_name}' not found in Colab Secrets.\n"
                "  Falling back to manual entry."
            )

    if not api_key:
        api_key = getpass.getpass(
            f"  Enter your Groq API key for '{secret_name}': "
        ).strip()

    if not api_key:
        raise ValueError(
            "No Groq API key provided. "
            f"Add it as a Colab Secret named '{secret_name}' "
            "or enter it when prompted."
        )

    print(f"  Key loaded : gsk_...{api_key[-4:]}")
    return api_key


GROQ_API_KEY: str = load_groq_api_key(GROQ_SECRET_NAME)


# =============================================================================
# F.  *** PROMPT ***
#
#  The ctx dict provides:
#    ctx["title"]        — course title
#    ctx["organization"] — course provider
#    ctx["description"]  — cleaned course description
#
#  The model must output EXACTLY one of: Beginner | Intermediate | Advanced
# =============================================================================


def _build_prompt(ctx: Dict[str, str]) -> str:
    """Return the filled prompt string for difficulty prediction.

    Replace the body of this function with your own prompt technique.

    Args:
        ctx: Course context dict from get_course_context().

    Returns:
        Fully rendered prompt string ready for the LLM.
    """
    return f"""You are a course difficulty classification system.

            Determine whether the following course description is Beginner, Intermediate, or Advanced based on the complexity of concepts, prerequisites, tools, and technical depth.

            Output only one label.
            Respond with only the difficulty label - one word,
            no punctuation, no explanation.

            Description: {ctx['description']}

            Difficulty:
            """


print(f"\nPrompt template ready: {TECHNIQUE} / {TASK_TYPE}")


# =============================================================================
# G.  LLM CALLER
# =============================================================================

TPD_ERROR_PHRASES: Tuple[str, ...] = (
    "tokens per day",
    "tpd",
    "on_demand",
    "daily token",
    "token quota",
    "rate limit",
)

RETRY_BACKOFF_MIN: int = 15
RETRY_BACKOFF_MAX: int = 30


class TokenLimitExceeded(Exception):
    """Raised when the Groq daily token quota (TPD) is exhausted.

    No retry is attempted upon catching this exception.
    The caller is responsible for saving any partial results.
    """


def is_token_limit_error(exc: Exception) -> bool:
    """Return True if the exception signals a Groq daily TPD quota error.

    Args:
        exc: Exception raised by the OpenAI client.

    Returns:
        True if any TPD-related phrase is found in the error message.
    """
    return any(phrase in str(exc).lower() for phrase in TPD_ERROR_PHRASES)


def normalise_label(raw: str) -> str:
    """Normalise a raw LLM response to a canonical difficulty label.

    Strips punctuation, whitespace and think-blocks; performs case-insensitive
    matching against VALID_LABELS. Returns 'Unknown' if no match is found.

    Args:
        raw: Raw string output from the LLM.

    Returns:
        Canonical label string from VALID_LABELS, or 'Unknown'.
    """
    # Strip Qwen3-style <think>...</think> blocks
    cleaned = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL).strip()
    cleaned = re.sub(r"[^a-zA-Z\s]", "", cleaned).strip()

    for label in VALID_LABELS:
        if label.lower() in cleaned.lower():
            return label

    # Fallback: first capitalised word
    words = cleaned.split()
    for word in words:
        word_clean = word.strip().capitalize()
        if word_clean in VALID_LABELS:
            return word_clean

    return "Unknown"


def call_llm(
    prompt: str,
    model_name: str,
    max_tokens: int = MAX_TOKENS,
    top_p: float = TOP_P,
    retries: int = 3,
) -> Tuple[str, float]:
    """Call the Groq API and return (response_text, latency_seconds).

    Uses the module-level GROQ_API_KEY and TEMPERATURE.
    Strips <think>...</think> blocks from Qwen3 outputs.
    Raises TokenLimitExceeded immediately on daily quota errors.
    Retries up to `retries` times with random back-off on transient errors.

    Args:
        prompt:     Fully rendered user prompt string.
        model_name: Groq model identifier string.
        max_tokens: Maximum tokens in the completion.
        top_p:      Nucleus sampling parameter.
        retries:    Number of retry attempts on transient errors.

    Returns:
        Tuple of (normalised_label, latency_in_seconds).

    Raises:
        TokenLimitExceeded: If a daily TPD quota error is detected.
    """
    client = OpenAI(api_key=GROQ_API_KEY, base_url=GROQ_BASE_URL)
    messages = [{"role": "user", "content": prompt}]

    delay = random.uniform(DELAY_MIN, DELAY_MAX)
    print(f"      Waiting {delay:.0f}s...", end=" ", flush=True)
    time.sleep(delay)

    for attempt in range(1, retries + 1):
        t_start = time.perf_counter()
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=messages,
                temperature=TEMPERATURE,
                top_p=top_p,
                max_tokens=max_tokens,
            )
            latency = round(time.perf_counter() - t_start, 3)
            raw_text = response.choices[0].message.content or ""
            label = normalise_label(raw_text)
            print(f"OK → '{label}' ({latency}s)")
            return label, latency

        except Exception as exc:
            latency = round(time.perf_counter() - t_start, 3)
            if is_token_limit_error(exc):
                raise TokenLimitExceeded(str(exc)) from exc
            print(f"\n      Attempt {attempt}/{retries} failed: {exc}")
            if attempt < retries:
                backoff = random.uniform(RETRY_BACKOFF_MIN, RETRY_BACKOFF_MAX)
                print(f"      Retrying in {backoff:.0f}s...")
                time.sleep(backoff)
            else:
                print("      All retries exhausted. Recording empty output.")
                return "Unknown", latency


print("LLM caller ready.")


# =============================================================================
# H.  RUN EXPERIMENT
# =============================================================================

PROMPT_SNIPPET_LENGTH: int = 300


def build_result_record(
    run: int,
    model_name: str,
    idx: int,
    ctx: Dict[str, str],
    predicted_label: str,
    latency: float,
    prompt: str,
) -> Dict:
    """Assemble a single result record dict for storage in the results list.

    Args:
        run:             Run number (1-indexed).
        model_name:      Groq model identifier.
        idx:             Course row index in the dataset.
        ctx:             Course context dictionary.
        predicted_label: Normalised difficulty label from LLM.
        latency:         API call latency in seconds.
        prompt:          Full prompt string (truncated for storage).

    Returns:
        Dictionary with all result fields.
    """
    prompt_clean = re.sub(r"\s+", " ", prompt).strip()
    gold = ctx.get("gold_difficulty", "")
    is_correct = int(predicted_label == gold) if gold and gold != "nan" else None
    return {
        "run":               run,
        "model":             model_name,
        "task_type":         TASK_TYPE,
        "technique":         TECHNIQUE,
        "temperature":       TEMPERATURE,
        "top_p":             TOP_P,
        "course_idx":        idx,
        "course_title":      ctx["title"],
        "description":       ctx["description"],
        "gold_difficulty":   gold,
        "predicted_label":   predicted_label,
        "is_correct":        is_correct,
        "latency_seconds":   latency,
        "prompt_snippet":    prompt_clean[:PROMPT_SNIPPET_LENGTH] + "...",
        "timestamp":         datetime.now(timezone.utc).isoformat(),
    }


def print_token_limit_summary(
    all_records: List[Dict], exc: TokenLimitExceeded
) -> None:
    """Print a structured summary when the daily token limit is hit.

    Args:
        all_records: Records collected before the limit was reached.
        exc:         The TokenLimitExceeded exception that was raised.
    """
    last = all_records[-1] if all_records else {}
    print("\n" + "!" * SEPARATOR_LG)
    print("*** TOKEN LIMIT REACHED — experiment stopped here ***")
    print(f"  Model             : {last.get('model', 'N/A')}")
    print(f"  Task Type         : {last.get('task_type', 'N/A')}")
    print(f"  Technique         : {last.get('technique', 'N/A')}")
    print(f"  Run               : {last.get('run', 'N/A')}")
    print(f"  Course            : {last.get('course_title', 'N/A')}")
    print(f"  Records collected : {len(all_records)}")
    print(f"  Error             : {exc}")
    print("!" * SEPARATOR_LG)


all_records: List[Dict] = []
token_limit_hit: bool = False
experiment_ts: str = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

print("\n" + "=" * SEPARATOR_LG)
print("STARTING EXPERIMENT")
print("=" * SEPARATOR_LG)

try:
    for run in range(1, NUM_RUNS + 1):
        print(f"\n{'=' * SEPARATOR_LG}")
        print(f"RUN {run} / {NUM_RUNS}")
        print(f"{'=' * SEPARATOR_LG}")
        print(f"\n  [Task: {TASK_TYPE.upper()}]  temperature={TEMPERATURE}")

        for model_name in MODELS:
            print(f"\n    Model: {model_name}")
            print(f"\n      Technique: {TECHNIQUE}")

            for idx, row in DATASET_DF.iterrows():
                ctx = get_course_context(row)
                prompt = _build_prompt(ctx)
                print(
                    f"      [{idx:02d}] {ctx['title'][:50]}...",
                    end=" ",
                )

                predicted_label, latency = call_llm(
                    prompt,
                    model_name,
                    max_tokens=MAX_TOKENS,
                    top_p=TOP_P,
                )

                all_records.append(
                    build_result_record(
                        run, model_name, idx, ctx,
                        predicted_label, latency, prompt
                    )
                )

except TokenLimitExceeded as exc:
    token_limit_hit = True
    print_token_limit_summary(all_records, exc)

RESULTS_DF: pd.DataFrame = pd.DataFrame(all_records)
experiment_status = "STOPPED EARLY (token limit)" if token_limit_hit else "COMPLETE"
print(f"\nExperiment {experiment_status}. Total records: {len(RESULTS_DF)}")


# =============================================================================
# I.  EVALUATION METRICS
# =============================================================================
#
# Four primary classification metrics:
#   1. Accuracy            — fraction of exactly correct predictions
#   2. Macro F1            — unweighted mean F1 across all three classes
#   3. Cohen's Kappa       — agreement adjusted for chance (ordinal classes)
#   4. MCC                 — Matthews Correlation Coefficient (balanced metric)
#
# Supporting metrics:
#   - Precision / Recall (macro)
#   - Off-by-one accuracy (adjacent-label tolerance)
#   - Label distribution: predicted vs gold
#
# Latency composite:
#   - avg / min / max / total latency per (run, model)
#   - throughput_courses_per_min = total_samples / (total_latency / 60)
#   - latency_composite  = harmonic mean of (accuracy / avg_latency_sec)
#                          after per-metric normalisation — rewards models
#                          that are both accurate AND fast.
# =============================================================================

# Weights for the overall composite score
COMPOSITE_WEIGHTS: Dict[str, float] = {
    "accuracy":      0.30,
    "macro_f1":      0.30,
    "cohen_kappa":   0.25,
    "mcc":           0.15,
}


def off_by_one_accuracy(
    y_true: List[str], y_pred: List[str]
) -> float:
    """Compute off-by-one accuracy using the ordinal label order.

    A prediction is counted as 'acceptable' if it is within one ordinal
    step of the gold label (e.g. Beginner ↔ Intermediate is acceptable;
    Beginner ↔ Advanced is not).

    Args:
        y_true: List of gold difficulty labels.
        y_pred: List of predicted difficulty labels.

    Returns:
        Off-by-one accuracy in [0, 1], rounded to 4 decimal places.
    """
    if not y_true:
        return 0.0
    correct = 0
    for gt, gp in zip(y_true, y_pred):
        oi_gt = LABEL_ORDER.get(gt, -1)
        oi_gp = LABEL_ORDER.get(gp, -1)
        if oi_gt >= 0 and oi_gp >= 0 and abs(oi_gt - oi_gp) <= 1:
            correct += 1
    return round(correct / len(y_true), 4)


def compute_label_distribution(labels: List[str]) -> Dict[str, int]:
    """Return count of each valid label in the list.

    Args:
        labels: List of label strings.

    Returns:
        Dictionary mapping each VALID_LABEL to its count.
    """
    counter = Counter(labels)
    return {label: counter.get(label, 0) for label in VALID_LABELS + ["Unknown"]}


def compute_latency_composite(
    accuracy: float, avg_latency: float
) -> float:
    """Compute a latency-accuracy composite score.

    Rewards models that are both accurate (higher is better) and fast
    (lower latency is better). Uses the formula:
        composite = accuracy / (1 + avg_latency_seconds)

    The denominator dampens the score for slow models without penalising
    very fast ones disproportionately.

    Args:
        accuracy:    Accuracy score in [0, 1].
        avg_latency: Mean API latency in seconds.

    Returns:
        Latency composite score, rounded to 4 decimal places.
    """
    if avg_latency < 0:
        return 0.0
    return round(accuracy / (1.0 + avg_latency), 4)


def evaluate_group(
    group_df: pd.DataFrame,
) -> Dict:
    """Compute all evaluation metrics for a single (run, model) group.

    Args:
        group_df: Subset of RESULTS_DF for one (run, model) combination.

    Returns:
        Dictionary of all metric values for this group.
    """
    # Filter rows where gold label is available and valid
    valid = group_df[
        group_df["gold_difficulty"].notna()
        & group_df["gold_difficulty"].isin(VALID_LABELS)
        & group_df["predicted_label"].isin(VALID_LABELS)
    ].copy()

    n_total = len(group_df)
    n_valid = len(valid)
    n_unknown = (group_df["predicted_label"] == "Unknown").sum()

    if n_valid == 0:
        # No valid pairs — return NaN metrics
        return {
            "total_samples":      n_total,
            "valid_samples":      0,
            "unknown_preds":      int(n_unknown),
            "accuracy":           float("nan"),
            "macro_f1":           float("nan"),
            "macro_precision":    float("nan"),
            "macro_recall":       float("nan"),
            "cohen_kappa":        float("nan"),
            "mcc":                float("nan"),
            "off_by_one_acc":     float("nan"),
            "composite_score":    float("nan"),
            "avg_latency_sec":    group_df["latency_seconds"].mean(),
            "min_latency_sec":    group_df["latency_seconds"].min(),
            "max_latency_sec":    group_df["latency_seconds"].max(),
            "total_latency_sec":  group_df["latency_seconds"].sum(),
            "throughput_per_min": float("nan"),
            "latency_composite":  float("nan"),
        }

    y_true = valid["gold_difficulty"].tolist()
    y_pred = valid["predicted_label"].tolist()

    # Core classification metrics
    acc     = round(accuracy_score(y_true, y_pred), 4)
    mac_f1  = round(f1_score(y_true, y_pred, average="macro", zero_division=0), 4)
    mac_pre = round(precision_score(y_true, y_pred, average="macro", zero_division=0), 4)
    mac_rec = round(recall_score(y_true, y_pred, average="macro", zero_division=0), 4)

    # Cohen's Kappa (ordinal weighting — linear)
    try:
        kappa = round(cohen_kappa_score(y_true, y_pred, weights="linear"), 4)
    except Exception:
        kappa = float("nan")

    # Matthews Correlation Coefficient
    try:
        mcc_val = round(matthews_corrcoef(y_true, y_pred), 4)
    except Exception:
        mcc_val = float("nan")

    # Off-by-one accuracy
    obo = off_by_one_accuracy(y_true, y_pred)

    # Weighted composite score (accuracy + F1 + kappa + MCC)
    w = COMPOSITE_WEIGHTS
    kappa_norm = (kappa + 1) / 2 if not math.isnan(kappa) else 0.0  # kappa in [-1,1] → [0,1]
    mcc_norm   = (mcc_val + 1) / 2 if not math.isnan(mcc_val) else 0.0
    composite  = round(
        acc     * w["accuracy"]
        + mac_f1  * w["macro_f1"]
        + kappa_norm * w["cohen_kappa"]
        + mcc_norm   * w["mcc"],
        4,
    )

    # Latency stats
    avg_lat    = round(group_df["latency_seconds"].mean(), 4)
    min_lat    = round(group_df["latency_seconds"].min(), 4)
    max_lat    = round(group_df["latency_seconds"].max(), 4)
    total_lat  = round(group_df["latency_seconds"].sum(), 4)
    throughput = round(n_total / (total_lat / 60), 2) if total_lat > 0 else float("nan")
    lat_comp   = compute_latency_composite(acc, avg_lat)

    return {
        "total_samples":      n_total,
        "valid_samples":      n_valid,
        "unknown_preds":      int(n_unknown),
        "accuracy":           acc,
        "macro_f1":           mac_f1,
        "macro_precision":    mac_pre,
        "macro_recall":       mac_rec,
        "cohen_kappa":        kappa,
        "mcc":                mcc_val,
        "off_by_one_acc":     obo,
        "composite_score":    composite,
        "avg_latency_sec":    avg_lat,
        "min_latency_sec":    min_lat,
        "max_latency_sec":    max_lat,
        "total_latency_sec":  total_lat,
        "throughput_per_min": throughput,
        "latency_composite":  lat_comp,
    }


def evaluate(results_df: pd.DataFrame) -> pd.DataFrame:
    """Compute and aggregate all evaluation metrics from the results DataFrame.

    Groups by (run, model) and computes all classification metrics,
    latency stats, and composite scores.

    Args:
        results_df: Raw results DataFrame from the experiment loop.

    Returns:
        Aggregated metrics DataFrame sorted by composite_score descending.
    """
    if results_df.empty:
        print("No results to evaluate.")
        return pd.DataFrame()

    print("\n" + "=" * SEPARATOR_LG)
    print("COMPUTING EVALUATION METRICS")
    print("=" * SEPARATOR_LG)

    metric_rows: List[Dict] = []

    for (run, model), group_df in results_df.groupby(["run", "model"]):
        print(f"  Run {run} | {model} — {len(group_df)} samples")
        metrics = evaluate_group(group_df)
        metrics["run"]              = run
        metrics["model"]            = model
        metrics["technique"]        = TECHNIQUE
        metrics["task_type"]        = TASK_TYPE
        metrics["temperature"]      = TEMPERATURE
        metric_rows.append(metrics)

    metrics_df = pd.DataFrame(metric_rows)

    # Reorder columns for readability
    front_cols = [
        "run", "model", "technique", "task_type", "temperature",
        "total_samples", "valid_samples", "unknown_preds",
        "accuracy", "macro_f1", "macro_precision", "macro_recall",
        "cohen_kappa", "mcc", "off_by_one_acc",
        "composite_score",
        "avg_latency_sec", "min_latency_sec", "max_latency_sec",
        "total_latency_sec", "throughput_per_min", "latency_composite",
    ]
    available = [c for c in front_cols if c in metrics_df.columns]
    metrics_df = metrics_df[available]

    return metrics_df.sort_values(
        "composite_score", ascending=False
    ).reset_index(drop=True)


def print_per_class_report(results_df: pd.DataFrame) -> None:
    """Print sklearn classification report per model.

    Args:
        results_df: Raw results DataFrame.
    """
    print("\n" + "=" * SEPARATOR_LG)
    print("PER-CLASS CLASSIFICATION REPORT")
    print("=" * SEPARATOR_LG)

    for model, group_df in results_df.groupby("model"):
        valid = group_df[
            group_df["gold_difficulty"].isin(VALID_LABELS)
            & group_df["predicted_label"].isin(VALID_LABELS)
        ]
        if valid.empty:
            print(f"\n  {model}: no valid predictions")
            continue
        print(f"\n  Model: {model}")
        print(
            classification_report(
                valid["gold_difficulty"],
                valid["predicted_label"],
                labels=VALID_LABELS,
                zero_division=0,
            )
        )


def print_confusion_matrices(results_df: pd.DataFrame) -> None:
    """Print a confusion matrix for each model.

    Args:
        results_df: Raw results DataFrame.
    """
    print("\n" + "=" * SEPARATOR_LG)
    print("CONFUSION MATRICES")
    print("=" * SEPARATOR_LG)

    for model, group_df in results_df.groupby("model"):
        valid = group_df[
            group_df["gold_difficulty"].isin(VALID_LABELS)
            & group_df["predicted_label"].isin(VALID_LABELS)
        ]
        if valid.empty:
            continue
        cm = confusion_matrix(
            valid["gold_difficulty"],
            valid["predicted_label"],
            labels=VALID_LABELS,
        )
        cm_df = pd.DataFrame(
            cm, index=[f"True_{l}" for l in VALID_LABELS],
            columns=[f"Pred_{l}" for l in VALID_LABELS]
        )
        print(f"\n  Model: {model}")
        print(cm_df.to_string())


METRICS_DF: pd.DataFrame = evaluate(RESULTS_DF)
print_per_class_report(RESULTS_DF)
print_confusion_matrices(RESULTS_DF)
print(f"\nMetrics computed: {len(METRICS_DF)} rows")


# =============================================================================
# J.  FULL METRICS TABLE
# =============================================================================

DISPLAY_COLUMNS: List[str] = [
    "run", "model", "technique", "task_type", "temperature",
    "total_samples", "valid_samples", "unknown_preds",
    "accuracy", "macro_f1", "macro_precision", "macro_recall",
    "cohen_kappa", "mcc", "off_by_one_acc",
    "composite_score",
    "avg_latency_sec", "throughput_per_min", "latency_composite",
]


def print_full_metrics_table(metrics_df: pd.DataFrame) -> None:
    """Configure pandas display and print the full metrics table.

    Args:
        metrics_df: Aggregated metrics DataFrame.
    """
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 260)
    pd.set_option("display.float_format", "{:.4f}".format)

    visible_cols = [col for col in DISPLAY_COLUMNS if col in metrics_df.columns]
    print("\n" + "=" * SEPARATOR_XL)
    print("FULL METRICS TABLE")
    print("=" * SEPARATOR_XL)
    print(metrics_df[visible_cols].to_string(index=False))


print_full_metrics_table(METRICS_DF)


# =============================================================================
# K.  FINAL SUMMARY — BEST & WORST MODEL
# =============================================================================

SUMMARY_METRIC_COLS: List[str] = [
    "accuracy", "macro_f1", "cohen_kappa", "mcc",
    "off_by_one_acc", "composite_score",
    "avg_latency_sec", "throughput_per_min", "latency_composite",
]


def build_best_worst_summary(metrics_df: pd.DataFrame) -> pd.DataFrame:
    """Build a DataFrame ranking best and worst models for this experiment.

    Averages composite_score across all runs before ranking.

    Args:
        metrics_df: Aggregated metrics DataFrame from evaluate().

    Returns:
        DataFrame with one BEST row and one WORST row.
    """
    if metrics_df.empty:
        return pd.DataFrame()

    valid_cols = [col for col in SUMMARY_METRIC_COLS if col in metrics_df.columns]
    grouped = (
        metrics_df
        .groupby("model")[valid_cols]
        .mean()
        .round(4)
        .reset_index()
        .sort_values("composite_score", ascending=False)
        .reset_index(drop=True)
    )

    best  = grouped.iloc[0].to_dict()
    worst = grouped.iloc[-1].to_dict()
    best["rank"]  = "BEST"
    worst["rank"] = "WORST"

    summary = pd.DataFrame([best, worst])
    summary.insert(0, "technique",  TECHNIQUE)
    summary.insert(1, "task_type",  TASK_TYPE)
    summary.insert(2, "temperature", TEMPERATURE)
    return summary


def print_best_worst_summary(summary_df: pd.DataFrame) -> None:
    """Print a formatted best-and-worst model summary table to stdout.

    Args:
        summary_df: DataFrame produced by build_best_worst_summary().
    """
    if summary_df.empty:
        print("No data for best/worst summary.")
        return

    print("\n" + "=" * SEPARATOR_XXL)
    print("FINAL SUMMARY — BEST & WORST MODEL")
    print(
        f"Technique: {TECHNIQUE}  |  Task: {TASK_TYPE}  |  Temperature: {TEMPERATURE}"
    )
    print("(composite_score averaged across all runs; higher = better)")
    print("=" * SEPARATOR_XXL)
    print(
        f"  {'Rank':<6} {'Model':<32} {'Composite':>9} {'Accuracy':>9} "
        f"{'MacroF1':>8} {'Kappa':>7} {'MCC':>7} "
        f"{'OBO_Acc':>8} {'AvgLat':>8} {'Thruput':>8} {'LatComp':>8}"
    )
    print("  " + "-" * TABLE_ROW_WIDTH)

    for _, row in summary_df.iterrows():
        rank_marker = ">>>" if row["rank"] == "BEST" else "   "

        def _f(key: str, fmt: str = ".4f") -> str:
            val = row.get(key, float("nan"))
            return f"{val:{fmt}}" if not (isinstance(val, float) and math.isnan(val)) else "  N/A  "

        print(
            f"  {rank_marker} {row['rank']:<3} "
            f"{row['model']:<32} "
            f"{_f('composite_score'):>9} "
            f"{_f('accuracy'):>9} "
            f"{_f('macro_f1'):>8} "
            f"{_f('cohen_kappa'):>7} "
            f"{_f('mcc'):>7} "
            f"{_f('off_by_one_acc'):>8} "
            f"{_f('avg_latency_sec'):>8} "
            f"{_f('throughput_per_min', '.2f'):>8} "
            f"{_f('latency_composite'):>8}"
        )

    print("\n" + "=" * SEPARATOR_XXL)
    print("KEY")
    print("  >>> BEST       = highest composite_score across all models")
    print("      WORST      = lowest composite_score across all models")
    print("  Composite      = weighted blend: 30% Accuracy + 30% MacroF1 + 25% Kappa + 15% MCC")
    print("  OBO_Acc        = off-by-one accuracy (within one ordinal step counts as correct)")
    print("  Kappa          = Cohen's Kappa (linearly weighted, accounts for chance)")
    print("  MCC            = Matthews Correlation Coefficient (-1 to +1, chance-corrected)")
    print("  Thruput        = courses processed per minute")
    print("  LatComp        = accuracy / (1 + avg_latency_sec)  — higher = faster AND more accurate")
    print("=" * SEPARATOR_XXL)


SUMMARY_DF: pd.DataFrame = build_best_worst_summary(METRICS_DF)
print_best_worst_summary(SUMMARY_DF)


# =============================================================================
# L.  SAVE & DOWNLOAD
# =============================================================================

_file_tag: str = f"{TECHNIQUE}_{TASK_TYPE}_{experiment_ts}"
RESULTS_FILENAME: str  = f"difficulty_results_{_file_tag}.csv"
METRICS_FILENAME: str  = f"difficulty_metrics_{_file_tag}.csv"
SUMMARY_FILENAME: str  = f"difficulty_best_worst_{_file_tag}.csv"


def save_results(
    results_df: pd.DataFrame,
    metrics_df: pd.DataFrame,
    summary_df: pd.DataFrame,
    in_colab: bool,
) -> None:
    """Save all result DataFrames to CSV and optionally trigger downloads.

    Args:
        results_df: Raw results from the experiment loop.
        metrics_df: Aggregated metrics DataFrame.
        summary_df: Best/worst summary DataFrame.
        in_colab:   True if running inside Google Colab.
    """
    results_df.to_csv(RESULTS_FILENAME, index=False)
    metrics_df.to_csv(METRICS_FILENAME, index=False)
    if not summary_df.empty:
        summary_df.to_csv(SUMMARY_FILENAME, index=False)

    print(f"\nSaved: {RESULTS_FILENAME}  ({len(results_df)} rows)")
    print(f"Saved: {METRICS_FILENAME}  ({len(metrics_df)} rows)")
    print(f"Saved: {SUMMARY_FILENAME}  ({len(summary_df)} rows)")

    if in_colab:
        print("\nDownloading files...")
        files.download(RESULTS_FILENAME)
        files.download(METRICS_FILENAME)
        files.download(SUMMARY_FILENAME)
        print("Downloads initiated.")
    else:
        print("\nFiles saved locally.")

    print("\nDone.")


save_results(RESULTS_DF, METRICS_DF, SUMMARY_DF, IN_COLAB)

LIBRARY VERSIONS (active runtime)
  openai             2.32.0
  pandas             2.2.2
  numpy              2.0.2
  scikit-learn       1.6.1
  matplotlib         3.10.0
  scipy              1.16.3
Experiment Configuration
----------------------------------------
  Technique          : instruction
  Task type          : difficulty_prediction
  Temperature        : 0.0
  Runs               : 1
  Models             : ['llama-3.1-8b-instant', 'qwen/qwen3-32b', 'openai/gpt-oss-20b', 'openai/gpt-oss-120b', 'llama-3.3-70b-versatile']
  Courses            : 25
  Valid labels       : ['Beginner', 'Intermediate', 'Advanced']
  Total API calls    : 125
  Est. delay time    : ~19.8 min
  top_p              : 0.9

----------------------------------------
Dataset Load
----------------------------------------
  Loading Coursera dataset from Google Drive...
  Please wait...

Mounted at /content/drive
  Found dataset at: /content/drive/MyDrive/research/data/coursera_clean_sample.csv
  Cleaning and fi

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloads initiated.

Done.
